# Run Experiment: Change Point Detection → PMF Prediction

This notebook demonstrates the full pipeline:
1. **Load** arrival-rate time series (Cox/M/1 process)
2. **Configure** prediction method (BOCPD + adaptive OLS)
3. **Detect** change points and predict arrival rates
4. **Compute** queue-length PMF via uniformization
5. **Compare** predicted PMF with simulation histogram (KL divergence)

---
## Parameters (edit this cell to switch scenarios)

In [ ]:
# ── Scenario parameters (EDIT HERE) ──────────────────────────────────
Z0 = 80              # Initial value of Z(t): 5 or 80
a = 0.3              # Mean-reversion speed
b = 120              # Long-term mean
delta = 1.0          # Noise magnitude (None if not in filename)
n_samples = 500      # Number of samples

# Queue model
mu = 100.0           # Service rate
m = 1                # Number of servers
t_eval = [1, 5]      # Time points at which to evaluate PMF

# Prediction method
METHOD = 'cpd_bayesian'   # 'bayesian' | 'kalman' | 'cpd' | 'cpd_bayesian'
USE_ADAPTIVE = True        # Use adaptive window + threshold
W_MIN, W_MAX = 3, 15      # Adaptive window bounds
ADAPTIVE_THRESHOLD = True  # Link threshold to volatility
THRESHOLD_BASE = 0.01      # Base alarm threshold
THRESHOLD_K = 2.0          # Volatility scaling factor
WINDOW_METHOD = 'rolling_std'  # 'rolling_std' or 'ewma'

# PMF computation
N_STATES = 100       # Max queue states for PMF

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os, glob

from src import (
    UnifiedPredictor, PredictionConfig, BOOLES_cpdPredictor,
    transient_distribution_piecewise,
    calculate_kl_divergence, compare_pmfs_kl, load_histogram,
)

%matplotlib inline
plt.rcParams.update({'figure.dpi': 120, 'font.size': 11})

## Step 1: Load Arrival Data

In [ ]:
# Build filename from parameters
if delta is not None:
    csv_name = f"initial_value_{Z0}_samples_{n_samples}_a_{a}_b_{b}_delta_{delta}.csv"
else:
    csv_name = f"initial_value_{Z0}_samples_{n_samples}_a_{a}_b_{b}.csv"

data_dir = "data_integrated/arrival_data"
csv_path = os.path.join(data_dir, csv_name)

if not os.path.exists(csv_path):
    # Fallback: try without delta
    csv_name_alt = f"initial_value_{Z0}_samples_{n_samples}_a_{a}_b_{int(b)}.csv"
    csv_path_alt = os.path.join(data_dir, csv_name_alt)
    if os.path.exists(csv_path_alt):
        csv_path = csv_path_alt
    else:
        print(f"File not found: {csv_path}")
        print("Available files:")
        for f in sorted(glob.glob(os.path.join(data_dir, f"initial_value_{Z0}*.csv")))[:10]:
            print(f"  {os.path.basename(f)}")

data = pd.read_csv(csv_path)
print(f"Loaded: {csv_path}")
print(f"  Shape: {data.shape}, Time range: [{data['time'].min():.2f}, {data['time'].max():.2f}]")
print(f"  Value range: [{data['value'].min():.2f}, {data['value'].max():.2f}]")

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(data['time'], data['value'], lw=0.8)
ax.set_title(f'Arrival Rate Z(t) — Z0={Z0}, a={a}, b={b}')
ax.set_xlabel('Time'); ax.set_ylabel('Z(t)')
ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

## Step 2: Change Point Detection + Prediction

In [ ]:
if USE_ADAPTIVE and METHOD == 'cpd_bayesian':
    # Use BOOLES_cpdPredictor directly with adaptive window + threshold
    predictor = BOOLES_cpdPredictor(
        hazard_lambda=50.0,
        mu0=float(b),       # Prior mean set to long-term mean
        kappa0=0.1,
        alpha0=2.0,
        beta0=5.0,
        alarm_threshold=THRESHOLD_BASE,
    )
    metrics = predictor.adaptive_rolling_prediction(
        data,
        w_min=W_MIN,
        w_max=W_MAX,
        plot=True,
        method='ols',
        file_name=csv_name,
        adaptive_threshold=ADAPTIVE_THRESHOLD,
        threshold_base=THRESHOLD_BASE,
        threshold_k=THRESHOLD_K,
        window_method=WINDOW_METHOD,
    )
    results_dict = {
        'predictor': predictor,
        'summary_metrics': metrics,
        'detailed_results': predictor.results,
    }
else:
    # Use UnifiedPredictor for non-adaptive or other methods
    cfg = PredictionConfig(
        method=METHOD,
        window_size=W_MIN,
        hazard_lambda=50.0,
        mu0=float(b),
        alarm_threshold=THRESHOLD_BASE,
        mu=mu,
        plot=True,
        verbose=True,
    )
    up = UnifiedPredictor(cfg)
    results_dict = up.predict(data)
    predictor = results_dict['predictor']
    metrics = results_dict['summary_metrics']

print("\n--- Summary Metrics ---")
if metrics:
    for k, v in metrics.items():
        if isinstance(v, float):
            print(f"  {k}: {v:.4f}")
        else:
            print(f"  {k}: {v}")

## Step 3: Extract Piecewise Arrival Rates for PMF

In [ ]:
# Extract step-function values and time intervals from prediction results
r = predictor.results
step_vals = np.array(r.get('predicted_step_function', r.get('stepwise_value', [])))
step_dt = np.array(r.get('predicted_step_function_time_interval', []))

# If stepwise_value is a Series on the DataFrame, extract from the data
if len(step_vals) == 0 and 'stepwise_value' in r:
    sv = r['stepwise_value']
    if hasattr(sv, 'dropna'):
        sv = sv.dropna()
    step_vals = np.array(sv)

# Build Z_piece and dt_piece from the stepwise prediction
# Group consecutive identical stepwise values into segments
if hasattr(step_vals, 'values'):
    step_vals = step_vals.values

# Use prediction times for dt
pred_times = np.array(r.get('prediction_times', data['time'].values))
if hasattr(pred_times, 'values'):
    pred_times = pred_times.values

# Simple approach: use stepwise values directly as Z_piece
# and compute dt from time differences
valid_mask = ~np.isnan(step_vals) if len(step_vals) > 0 else np.array([])
if valid_mask.sum() > 0:
    Z_piece = step_vals[valid_mask]
    t_valid = pred_times[valid_mask] if len(pred_times) == len(step_vals) else pred_times[:valid_mask.sum()]
    dt_piece = np.diff(t_valid, prepend=t_valid[0])
    dt_piece[0] = dt_piece[1] if len(dt_piece) > 1 else 0.02
    # Ensure positive dt
    dt_piece = np.where(dt_piece <= 0, 0.02, dt_piece)
    Z_piece = np.maximum(Z_piece, 1e-6)  # Ensure positive arrival rates
else:
    # Fallback: use raw data values
    Z_piece = np.maximum(data['value'].values, 1e-6)
    dt_piece = np.diff(data['time'].values, prepend=data['time'].values[0])
    dt_piece[0] = dt_piece[1] if len(dt_piece) > 1 else 0.02

print(f"Z_piece: {len(Z_piece)} segments, range [{Z_piece.min():.2f}, {Z_piece.max():.2f}]")
print(f"dt_piece: total horizon = {dt_piece.sum():.2f}")
print(f"Mean arrival rate: {Z_piece.mean():.2f}")

## Step 4: Compute PMF and Compare with Simulation

In [ ]:
hist_dir = "data_integrated/Simulation_histograms"
service_rate = 10 if Z0 == 5 else 100 if Z0 == 80 else int(mu)

for t in t_eval:
    print(f"\n{'='*60}")
    print(f"PMF at t = {t}")
    print(f"{'='*60}")

    # Compute predicted PMF
    t_target = min(t, dt_piece.sum() * 0.99)
    try:
        pmf_pred = transient_distribution_piecewise(
            Z_piece, dt_piece, mu=mu, m=m, t=t_target, N=N_STATES
        )
        print(f"  PMF computed: {len(pmf_pred)} states, sum={np.sum(pmf_pred):.6f}")
    except Exception as e:
        print(f"  PMF computation failed: {e}")
        continue

    # Load simulation histogram
    hist_filename = f"for_histogram_CoxM1_Z0{Z0}_serv{service_rate}_t{t}.pickle"
    hist_path = os.path.join(hist_dir, hist_filename)

    fig, ax = plt.subplots(figsize=(12, 6))

    if os.path.exists(hist_path):
        hist_data = load_histogram(hist_path)
        counts = hist_data['counts']
        bins = hist_data['bins']

        # Align bins starting from 0
        if bins[0] != 0:
            interval = max(int(bins[1] - bins[0]), 1)
            bins_before = np.arange(0, bins[0], interval)
            bins = np.concatenate([bins_before, bins])
            counts = np.concatenate([np.zeros(len(bins_before)), counts])

        # Plot histogram
        ax.bar(bins[:-1], counts,
               width=bins[1]-bins[0] if len(bins) > 1 else 1,
               alpha=0.4, color='lightblue', label=f'Simulation t={t}')

        # Compute KL divergence
        # Align histogram to integer bins for comparison
        bin_centers = (bins[:-1] + bins[1:]) / 2
        idx_int = np.floor(bin_centers + 0.5).astype(int)
        max_idx = max(idx_int.max(), N_STATES - 1)
        hist_aligned = np.zeros(max_idx + 1)
        for ci, idx in enumerate(idx_int):
            if 0 <= idx <= max_idx:
                hist_aligned[idx] += counts[ci]

        kl = calculate_kl_divergence(hist_aligned[:N_STATES], pmf_pred[:N_STATES])
        print(f"  KL(Simulation || Predicted) = {kl:.6f}")
    else:
        print(f"  Histogram not found: {hist_filename}")
        kl = None

    # Plot PMF
    ax.plot(np.arange(min(N_STATES, len(pmf_pred))), pmf_pred[:N_STATES],
            'r-o', markersize=3, lw=1.5, label='Predicted PMF')
    ax.set_title(f'Queue Length Distribution at t={t}\n'
                 f'Z0={Z0}, a={a}, b={b}, mu={mu}'
                 + (f', KL={kl:.4f}' if kl is not None else ''))
    ax.set_xlabel('Queue Length')
    ax.set_ylabel('Probability')
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

## Step 5: Batch Run — Multiple NPZ Scenarios from Simulation_code

Run this cell to iterate over all CoxM/1 simulation files and compute KL divergence for each.

In [ ]:
# Parse NPZ files from Simulation_code
npz_dir = "Simulation_code"
npz_files = sorted(glob.glob(os.path.join(npz_dir, "pmf_CoxM1_*.npz")))
print(f"Found {len(npz_files)} CoxM/1 simulation files\n")

batch_results = []

for npz_path in npz_files:
    fname = os.path.basename(npz_path)
    # Parse: pmf_CoxM1_Z0{z0}_serv{serv}_T{T}_t{t}_v2.npz
    parts = fname.replace('.npz', '').split('_')
    try:
        z0_str = [p for p in parts if p.startswith('Z0')][0]
        z0 = int(z0_str[2:])
        serv_str = [p for p in parts if p.startswith('serv')][0]
        serv = int(serv_str[4:])
        T_str = [p for p in parts if p.startswith('T') and p[1:].isdigit()][0]
        T_val = int(T_str[1:])
        t_str = [p for p in parts if p.startswith('t') and p[1:].isdigit() and not p.startswith('T')][0]
        t_val = int(t_str[1:])
    except (IndexError, ValueError):
        print(f"  Skipping {fname} — could not parse params")
        continue

    print(f"  {fname}: Z0={z0}, serv={serv}, T={T_val}, t={t_val}")

    # Load npz
    npz_data = np.load(npz_path)
    keys = list(npz_data.keys())

    batch_results.append({
        'file': fname,
        'Z0': z0,
        'service_rate': serv,
        'T': T_val,
        't': t_val,
        'npz_keys': keys,
    })

print(f"\nParsed {len(batch_results)} scenarios")
if batch_results:
    df_scenarios = pd.DataFrame(batch_results)
    print(df_scenarios[['file', 'Z0', 'service_rate', 'T', 't']].to_string(index=False))